In [ ]:
#----Strongly Connected Component (SCC) Condensation -- -Simplify a directed oncology workflow containing circular dependencies.
# Suppose laboratory review and oncology review depend on each other, as do treatment planning and pharmacy review. 
# SCC condensation groups each mutually reachable set into one node. This can help analyze complex workflow dependencies.

graph = {
    "Lab_Review"            : ["Oncology_Review"],
    "Oncology_Review"       : ["Lab_Review", "Treatment_Planning"],
    "Treatment_Planning"    : ["Pharmacy_Review"],
    "Pharmacy_Review"       : ["Treatment_Planning"],
    "Treatment_Approval"    : []
}
def find_sccs(graph):
    index = 0
    indices = {}
    low = {}
    stack = []
    on_stack = set()
    components = []
    def dfs(node):
        nonlocal index
        indices[node] = low[node] = index
        index += 1
        stack.append(node)
        on_stack.add(node)
        for neighbor in graph[node]:
            if neighbor not in indices:
                dfs(neighbor)
                low[node] = min(low[node], low[neighbor])
            elif neighbor in on_stack:
                low[node] = min(low[node], indices[neighbor])
        if low[node] == indices[node]:
            component = []
            while True:
                member = stack.pop()
                on_stack.remove(member)
                component.append(member)
                if member == node:
                    break
            components.append(component)
    for node in graph:
        if node not in indices:
            dfs(node)
    return components
components = find_sccs(graph)
for group in components:
    print("Workflow group:", group)
# Compress SCCs into a new directed graph.
component_of = {}
for i, group in enumerate(components):
    for node in group:
        component_of[node] = i
condensed = {i: set() for i in range(len(components))}         # -- {0: set(), 1: set(), 2: set()}
for node, neighbors in graph.items():
    for neighbor in neighbors:
        a = component_of[node]
        b = component_of[neighbor]
        if a != b:
            condensed[a].add(b)
print("\nCondensed workflow:")
for i, targets in condensed.items():
    print(components[i], "->", [components[j] for j in targets])

Workflow group: ['Pharmacy_Review', 'Treatment_Planning']
Workflow group: ['Oncology_Review', 'Lab_Review']
Workflow group: ['Treatment_Approval']

Condensed workflow:
['Pharmacy_Review', 'Treatment_Planning'] -> []
['Oncology_Review', 'Lab_Review'] -> [['Pharmacy_Review', 'Treatment_Planning']]
['Treatment_Approval'] -> []
